# Cross-Lingual Audio Deepfake Detection — End-to-End Pipeline

**Parv Shah (23BCE219) · Divya Patel (23BCE227)** — Dept. of CSE, Nirma University

Runs the four roadmap phases end to end using the `deepfake_detection` package:

| Phase | What it does | Output |
|---|---|---|
| 1 | Fetch + cache features from ASVspoof 2019 LA, IndicSynth, FLEURS | `cache/*.npz` |
| 2 | Train the Triple-Branch ResNeXt on English (ASVspoof LA) | baseline EER |
| 3 | Zero-shot evaluation on Gujarati and Hindi | the generalization gap |
| 4 | Fine-tune the MLP fusion layer on 10% of the Indic data | recovered EER |

The architecture follows Tahaoglu et al. [11] Table 1 (PDF p. 7) and its stated
hyperparameters (Adam, lr 1e-4, batch 32, checkpoint on best validation EER — [11] PDF p. 8).

> **No fabricated results.** Every number below is produced by the code in this notebook
> during the run. Where a quantity cannot be computed — min t-DCF, which needs an ASV
> back-end — it is recorded as *not computed* rather than estimated.

## 0. Setup

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os, sys, json, torch
# Run from the project root so `deepfake_detection` is importable.
if os.path.basename(os.getcwd()) == "deepfake_detection":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

print("cwd:", os.getcwd())
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("threads:", torch.get_num_threads())

## Phase 1 — Data and preprocessing

Fetches a **subsample** of each corpus and caches MFCC/LFCC/CQCC features to `cache/*.npz`.
Caching matters because the bottleneck is network I/O, not compute: the ASVspoof mirror
streams at roughly 7 rows/s.

Two things worth knowing about the ASVspoof loader:

1. The mirror's Parquet files are **sorted by label** — bonafide occupies row groups 0–25,
   spoof 25 onward. Sequential streaming would need ~1 hour to reach the first spoof clip,
   and a naive "first N rows" subsample would be 100% bonafide. The loader instead reads
   *selected row groups* from each label region over HTTP range requests.
2. Sampling is **balanced** (equal real/fake) rather than preserving the corpus's true
   1:8.8 ratio. EER is threshold-free and prior-independent, so this does not bias it.

In [ ]:
!python -m deepfake_detection.cache_data

In [ ]:
import json, os
man = json.load(open("cache/manifest.json", encoding="utf-8"))
print(json.dumps(man, indent=2))

## Phases 2–4 — Train, evaluate zero-shot, adapt

Runs base training on ASVspoof, zero-shot evaluation on both Indic languages, and
fusion-only fine-tuning. Writes `results/table_v_results.{json,md}`,
`results/training_log*.csv` and `checkpoints/*.pt`.

In [ ]:
!python -m deepfake_detection.run_phases

## Results — Table V

In [ ]:
import json
R = json.load(open("results/table_v_results.json", encoding="utf-8"))

base = R["phase2"]["eer_percent"]
print(f"Phase 2  English baseline EER : {base:.2f}%\n")
for lang in ("Gujarati", "Hindi"):
    z = R["phase3"][lang]; f = R["phase4"][lang]
    print(f"{lang}")
    print(f"  Phase 3 zero-shot EER      : {z['eer_percent']:.2f}%  "
          f"(+{z['eer_increase_absolute_pp']:.2f} pp, "
          f"{z['eer_increase_relative_percent']:+.1f}% relative to English)")
    print(f"  Phase 4 adapted EER        : {f['eer_percent']:.2f}%  "
          f"(recovered {f['eer_recovered_pp']:+.2f} pp)")
print("\nmin t-DCF:", R["min_tdcf_note"])

## Training curves

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, os

log = pd.read_csv("results/training_log.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(log.epoch, log.train_loss, marker="o", ms=3)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("training loss (BCE)")
ax[0].set_title("Phase 2 — training loss"); ax[0].grid(alpha=.3)
ax[1].plot(log.epoch, log.val_eer_percent, marker="o", ms=3, color="#d62728")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("validation EER (%)")
ax[1].set_title("Phase 2 — validation EER"); ax[1].grid(alpha=.3)
plt.tight_layout()
os.makedirs("results/figures", exist_ok=True)
plt.savefig("results/figures/phase2_curves.png", dpi=150)
plt.show()

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, os

fig, ax = plt.subplots(figsize=(6.5, 4))
for lang, colour in (("gujarati", "#1f77b4"), ("hindi", "#2ca02c")):
    p = f"results/training_log_finetune_{lang}.csv"
    if os.path.exists(p):
        d = pd.read_csv(p)
        ax.plot(d.epoch, d.val_eer_percent, marker="o", ms=3,
                color=colour, label=lang.capitalize())
ax.set_xlabel("fine-tuning epoch"); ax.set_ylabel("held-out EER (%)")
ax.set_title("Phase 4 — fusion-layer adaptation"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout()
plt.savefig("results/figures/phase4_curves.png", dpi=150)
plt.show()

## What these numbers mean — and what they don't

- **Single run, one seed.** No multi-seed averaging. Differences of a point or two
  between conditions are within the noise a different seed could produce. Multi-seed
  averaging is the first thing to add.
- **Subsampled and short.** Training used a fraction of ASVspoof 2019 LA for a fraction
  of [11]'s 100 epochs, on CPU. The Phase 2 EER is this run's number; it is not a
  reproduction of [11]'s 1.05%, and the gap to that figure is mostly budget, not method.
- **The Phase 3 comparison is the point.** The absolute EERs matter less than the
  *increase* from English to Gujarati/Hindi under one fixed decision boundary. That
  increase is what the project set out to quantify.
- **Indic evaluation sets are small** (600 clips per language) and their real and fake
  halves come from *different corpora* (FLEURS vs IndicSynth), so some of the measured
  separation may be corpus, not synthesis — the same confound the matched-pair notebook
  was built to control for. Phase 3 numbers should be read with that in mind.
- **min t-DCF is not reported** because no ASV back-end is available.